<a href="https://colab.research.google.com/github/ai-agent-yonsei/agent/blob/main/3.1%20Routing_OpenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-based Routing

**사용자의 질문을 LLM이 먼저 분석하고, 가장 적절한 Route를 선택한 뒤 해당 전문 역할로 답변하는 구조**  

- `Reasoning → Routing → Orchestration → Action` 흐름을 코드로 확인
- **OpenAI를 기본 Provider로 바로 실행**
- 필요하면 **Claude / Google Gemini / Hugging Face**로 교체 가능
- 모든 API Key는 **Google Colab Secrets**에서 안전하게 불러오기
> **Google Colab Secret 사용**   
권장 Secret 이름:
- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY` *(Claude 사용 시)*
- `GEMINI_API_KEY` *(Google Gemini 사용 시)*
- `HF_TOKEN` *(Hugging Face 사용 시)*


In [1]:
# ============================================================
#  1. 필요한 SDK 설치
# ============================================================

# ------------------------------------------------------------
# [기본 실행] OpenAI
# ------------------------------------------------------------
!pip install -q -U openai

# ------------------------------------------------------------
# [선택] Claude
# Claude를 사용할 때만 아래 주석 해제
# ------------------------------------------------------------
# !pip install -q -U anthropic

# ------------------------------------------------------------
# [선택] Google Gemini
# Gemini를 사용할 때만 아래 주석 해제
# ------------------------------------------------------------
# !pip install -q -U google-genai

# ------------------------------------------------------------
# [선택] Hugging Face
# Hugging Face Inference API를 사용할 때만 아래 주석 해제
# ------------------------------------------------------------
# !pip install -q -U huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.7 MB/s eta 0:00:00


In [2]:

from google.colab import userdata
import os

# ------------------------------------------------------------
# 2. [기본 사용] OpenAI API Key
# ------------------------------------------------------------
# Colab 왼쪽 메뉴 → Secrets(열쇠 아이콘) →
# 이름을 OPENAI_API_KEY 로 등록하고 Notebook access를 ON 하세요.

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY를 Colab Secret에서 불러왔습니다.")


# ------------------------------------------------------------
# [선택] Claude
# Claude를 사용할 때만 아래 주석 해제
# ------------------------------------------------------------

# ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
# os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
# print("ANTHROPIC_API_KEY를 Colab Secret에서 불러왔습니다.")


# ------------------------------------------------------------
# [선택] Google Gemini
# Gemini를 사용할 때만 아래 주석 해제
# ------------------------------------------------------------

# GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
# os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
# print("GEMINI_API_KEY를 Colab Secret에서 불러왔습니다.")


# ------------------------------------------------------------
# [선택] Hugging Face
# Hugging Face Inference API를 사용할 때만 아래 주석 해제
# ------------------------------------------------------------

# HF_TOKEN = userdata.get("HF_TOKEN")
# os.environ["HF_TOKEN"] = HF_TOKEN
# print("HF_TOKEN을 Colab Secret에서 불러왔습니다.")


OPENAI_API_KEY를 Colab Secret에서 불러왔습니다.


In [3]:
# ============================================================
# 3. 사용할 LLM Provider 설정
# ============================================================

# ------------------------------------------------------------
# [기본 실행] OpenAI
# ------------------------------------------------------------
PROVIDER = "openai"

# ------------------------------------------------------------
# [선택] Claude
# PROVIDER = "claude"
# ------------------------------------------------------------

# ------------------------------------------------------------
# [선택] Google Gemini
# PROVIDER = "gemini"
# ------------------------------------------------------------

# ------------------------------------------------------------
# [선택] Hugging Face
# PROVIDER = "huggingface"
# ------------------------------------------------------------


# ============================================================
# Provider별 모델 설정
# ============================================================

# [기본] OpenAI
OPENAI_MODEL = "gpt-5.6-luna"

# [선택] Claude
# 필요 시 계정에서 사용 가능한 모델명으로 변경
# CLAUDE_MODEL = "claude-sonnet-4-6"

# [선택] Google Gemini
# 필요 시 계정에서 사용 가능한 모델명으로 변경
# GEMINI_MODEL = "gemini-3.8-flash"

# [선택] Hugging Face
# Inference API에서 사용할 텍스트 생성 모델
# 아래 모델명은 예시이며, 필요에 따라 변경 가능
# HF_MODEL = "Qwen/Qwen2.5-7B-Instruct"


## Provider-independent LLM 호출 함수

Agent의 Routing 로직에서는 특정 회사 SDK를 직접 호출하지 않고 `call_llm()`만 사용

In [4]:
# ============================================================
# 4. 공통 LLM 호출 함수
# ============================================================

def call_llm(system_prompt: str, user_prompt: str) -> str:
    """
    Provider와 관계없이 동일한 방식으로 LLM을 호출하기 위한 함수.

    Parameters
    ----------
    system_prompt : str
        모델이 따라야 할 역할/규칙

    user_prompt : str
        실제 사용자 질문

    Returns
    -------
    str
        모델의 텍스트 응답
    """

    # ========================================================
    # 1) OpenAI  [기본 실행]
    # ========================================================
    if PROVIDER == "openai":
        from openai import OpenAI

        # OPENAI_API_KEY 환경변수를 자동 사용
        client = OpenAI()

        response = client.responses.create(
            model=OPENAI_MODEL,
            instructions=system_prompt,
            input=user_prompt,
            max_output_tokens=500
        )

        return response.output_text


    # ========================================================
    # 2) Claude  [선택]
    # 사용하려면 Claude 관련 주석을 해제
    # ========================================================
    elif PROVIDER == "claude":
        from anthropic import Anthropic

        client = Anthropic(
            api_key=os.environ["ANTHROPIC_API_KEY"]
        )

        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=1200,
            system=system_prompt,
            messages=[
                {
                    "role": "user",
                    "content": user_prompt,
                }
            ],
        )

        return response.content[0].text


    # ========================================================
    # 3) Google Gemini  [선택]
    # 사용하려면 Gemini 관련 주석을 해제하세요.
    # ========================================================
    elif PROVIDER == "gemini":
        from google import genai
        from google.genai import types

        client = genai.Client(
            api_key=os.environ["GEMINI_API_KEY"]
        )

        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt
            ),
        )

        return response.text


    # ========================================================
    # 4) Hugging Face Inference API  [선택]
    # 사용하려면 Hugging Face 관련 주석을 해제하세요.
    # ========================================================
    elif PROVIDER == "huggingface":
        from huggingface_hub import InferenceClient

        client = InferenceClient(
            model=HF_MODEL,
            token=os.environ["HF_TOKEN"]
        )

        # OpenAI/Claude/Gemini처럼 system/user 역할을 분리할 수 있도록
        # chat_completion 형태로 호출
        response = client.chat_completion(
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            max_tokens=1200,
            temperature=0.2,
        )

        return response.choices[0].message.content


    else:
        raise ValueError(
            f"지원하지 않는 PROVIDER입니다: {PROVIDER}"
        )


## Route 정의


| Route | 역할 |
|---|---|
| `CODE` | 프로그래밍/코드 |
| `DATA` | 데이터/AI/ML |
| `WRITING` | 글쓰기/문서/번역 |
| `GENERAL` | 일반 질문 |

중요한 점은 **Router가 답변을 직접 만드는 것이 아니라, 어느 경로로 보낼지를 판단한다는 것**


In [5]:
# ============================================================
# 5. Route 정의
# ============================================================

ROUTES = {

    "CODE": """
당신은 Python 및 소프트웨어 개발 전문가입니다.

사용자의 질문에 대해:
- 코드를 명확하게 설명하고
- 필요한 경우 실행 가능한 코드 예제를 제공하고
- 오류 가능성이 있다면 원인까지 설명하세요.

답변은 한국어로 작성하세요.
""",

    "DATA": """
당신은 데이터 분석, 머신러닝, 딥러닝, 생성형 AI 전문가입니다.

사용자의 질문을 데이터 및 AI 관점에서 분석하세요.

가능하면 다음 순서로 쉽게 설명하세요.
1. 개념
2. 원리
3. 실제 예시

답변은 한국어로 작성하세요.
""",

    "WRITING": """
당신은 전문적인 글쓰기 및 문서 작성 전문가입니다.

사용자의 요청에 맞게:
- 문장을 자연스럽게 구성하고
- 목적과 독자를 고려하며
- 필요한 경우 표현을 전문적으로 다듬으세요.

답변은 한국어로 작성하세요.
""",

    "GENERAL": """
당신은 정확하고 이해하기 쉬운 범용 AI Assistant입니다.

사용자의 질문에 간결하고 정확하게 답변하세요.

답변은 한국어로 작성하세요.
"""
}


## LLM Router

Router에게 사용자 질문을 주고:

> **“이 질문은 어느 Route로 보내는 것이 가장 적절한가?”**

를 판단하도록 함

Rule-based Routing처럼 `if "python" in question:`으로 결정하지 않고, **LLM이 질문의 의미와 의도(Intent)를 보고 Route를 선택**


In [6]:
# ============================================================
#  6. LLM-based Router
# ============================================================

def route_question(question: str) -> str:
    """
    사용자 질문을 LLM이 분석하여
    CODE / DATA / WRITING / GENERAL 중 하나를 선택한다.

    이 함수가 LLM-based Routing의 핵심이다.
    """

    router_system_prompt = """
당신은 AI Agent의 Router입니다.

사용자의 질문의 '핵심 의도(Intent)'를 분석한 뒤
아래 네 개의 Route 중 가장 적절한 하나를 선택하세요.

[CODE]
- 프로그래밍
- Python
- 소프트웨어 개발
- 코드 작성/수정
- 오류 해결
- 알고리즘

[DATA]
- 데이터 분석
- 머신러닝
- 딥러닝
- 생성형 AI
- 통계
- 데이터베이스
- AI 개념 설명

[WRITING]
- 글쓰기
- 문장 수정
- 이메일 작성
- 보고서 작성
- 번역
- 요약
- 표현 개선

[GENERAL]
- 위 세 가지에 명확하게 속하지 않는 일반 질문

중요한 판단 원칙:
- 특정 단어 하나만 보고 분류하지 마세요.
- 사용자가 '궁극적으로 무엇을 해달라고 하는지'를 기준으로 판단하세요.
- 예: "AI 프로젝트 제안서의 첫 문장을 써줘"는 AI라는 단어가 있어도
      핵심 목적이 글쓰기이므로 WRITING입니다.

반드시 아래 네 단어 중 하나만 출력하세요.

CODE
DATA
WRITING
GENERAL

다른 설명이나 이유는 출력하지 마세요.
"""

    # --------------------------------------------------------
    # Reasoning:
    # LLM이 질문의 의미/의도를 보고 적절한 경로를 판단
    # --------------------------------------------------------
    route = call_llm(
        system_prompt=router_system_prompt,
        user_prompt=question
    )

    # 출력의 공백/줄바꿈 제거
    route = route.strip().upper()

    # 예상하지 못한 출력이 오면 GENERAL로 fallback
    if route not in ROUTES:
        print(f"예상하지 못한 Routing 결과: {route}")
        route = "GENERAL"

    return route


##  전체 Agent 실행

전체 흐름은 다음과 같습니다.

```text
User Question
     ↓
Reasoning
"질문의 핵심 의도가 무엇인가?"
     ↓
Routing
CODE / DATA / WRITING / GENERAL
     ↓
Orchestration
선택된 Route의 System Prompt 호출
     ↓
Action
해당 전문 역할로 답변 생성
```


In [7]:
# ============================================================
#  7. 전체 Routed Agent 실행
# ============================================================

def run_agent(question: str):
    """
    1. 사용자 질문 입력
    2. Router LLM이 질문의 의도 판단
    3. Route 선택
    4. 선택된 전문 역할 실행
    5. 최종 답변 반환
    """

    print("=" * 70)
    print("USER QUESTION")
    print("=" * 70)
    print(question)

    # --------------------------------------------------------
    # 1) Reasoning + Routing
    # --------------------------------------------------------
    route = route_question(question)

    print("\n" + "=" * 70)
    print("LLM ROUTING DECISION")
    print("=" * 70)
    print(f"→ {route}")

    # --------------------------------------------------------
    # 2) Orchestration
    # 선택된 Route에 연결된 System Prompt 선택
    # --------------------------------------------------------
    selected_system_prompt = ROUTES[route]

    # --------------------------------------------------------
    # 3) Action
    # 선택된 전문 역할로 실제 답변 생성
    # --------------------------------------------------------
    answer = call_llm(
        system_prompt=selected_system_prompt,
        user_prompt=question
    )

    print("\n" + "=" * 70)
    print(f"RESPONSE FROM [{route}] ROUTE")
    print("=" * 70)
    print(answer)

    return {
        "question": question,
        "route": route,
        "answer": answer,
    }


##  8. 단일 질문 테스트

먼저 명확한 Coding 질문을 넣어보자

In [8]:
result = run_agent(
    "Python에서 pandas DataFrame의 결측값을 제거하는 코드를 알려줘."
)


USER QUESTION
Python에서 pandas DataFrame의 결측값을 제거하는 코드를 알려줘.

LLM ROUTING DECISION
→ CODE

RESPONSE FROM [CODE] ROUTE
Python의 pandas에서는 `dropna()`를 사용해 결측값(`NaN`, `None`)이 포함된 행이나 열을 제거할 수 있습니다.

```python
import pandas as pd

df = pd.DataFrame({
    "이름": ["철수", "영희", None, "민수"],
    "나이": [20, None, 25, 30],
    "점수": [90, 85, None, 95]
})

print(df)
```

### 1. 결측값이 하나라도 있는 행 제거

```python
df_clean = df.dropna()

print(df_clean)
```

결측값이 포함된 행 전체가 제거됩니다.

### 2. 결측값이 하나라도 있는 열 제거

```python
df_clean = df.dropna(axis=1)

print(df_clean)
```

### 3. 특정 열을 기준으로 결측 행 제거

```python
df_clean = df.dropna(subset=["나이"])

print(df_clean)
```

`나이` 열에 결측값이 있는 행만 제거합니다.

### 4. 모든 값이 결측값인 행만 제거

```python
df_clean = df.dropna(how="all")
```

기본값은 `how="any"`이며, 하나라도 결측값이 있으면 제거합니다.

### 5. 결측값이 일정 개수 이상 있는 행 제거

예를 들어, 값이 2개 이상 존재하는 행만 남기려면 다음과 같이 작성합니다.

```python
df_clean = df.dropna(thresh=2)
```

### 6. 원본 DataFrame 자체를 수정

```python
df.dropna(inplace=True)
```

`inplace=True`를 사용하면 별도의 변

## 여러 질문으로 Routing 비교

마지막 질문처럼 **여러 영역의 단어가 섞인 질문**을 넣어보면 LLM Routing의 장점이 더 잘 보임

예:

> `AI 프로젝트 제안서의 첫 문장을 전문적으로 작성해줘.`

`AI`라는 단어가 있으므로 단순 키워드 Router라면 DATA로 보낼 수도 있지만, 사용자의 실제 목적은 **문장 작성**이므로 WRITING이 더 적절


In [9]:
# ============================================================
#   9. 여러 질문을 순차적으로 테스트
# ============================================================

test_questions = [
    "Python에서 리스트를 정렬하는 코드를 만들어줘.",
    "머신러닝에서 overfitting이 발생하는 이유는 뭐야?",
    "교수님께 과제 제출이 늦어서 죄송하다는 이메일을 작성해줘.",
    "프랑스의 수도는 어디야?",
    "AI 프로젝트 제안서의 첫 문장을 전문적으로 작성해줘.",
]


for question in test_questions:
    result = run_agent(question)
    print("\n\n")


USER QUESTION
Python에서 리스트를 정렬하는 코드를 만들어줘.

LLM ROUTING DECISION
→ CODE

RESPONSE FROM [CODE] ROUTE
Python에서 리스트를 정렬하는 방법은 `sort()`와 `sorted()`가 있습니다.

## 1. `sort()` 사용하기

기존 리스트 자체를 정렬합니다.

```python
numbers = [5, 2, 9, 1, 7]

numbers.sort()

print(numbers)
```

실행 결과:

```python
[1, 2, 5, 7, 9]
```

내림차순으로 정렬하려면 `reverse=True`를 사용합니다.

```python
numbers = [5, 2, 9, 1, 7]

numbers.sort(reverse=True)

print(numbers)
```

결과:

```python
[9, 7, 5, 2, 1]
```

## 2. `sorted()` 사용하기

기존 리스트는 유지하고, 정렬된 새로운 리스트를 반환합니다.

```python
numbers = [5, 2, 9, 1, 7]

sorted_numbers = sorted(numbers)

print("기존 리스트:", numbers)
print("정렬된 리스트:", sorted_numbers)
```

결과:

```python
기존 리스트: [5, 2, 9, 1, 7]
정렬된 리스트: [1, 2, 5, 7, 9]
```

## 3. 문자열 리스트 정렬하기

```python
names = ["Charlie", "Alice", "Bob"]

names.sort()

print(names)
```

결과:

```python
['Alice', 'Bob', 'Charlie']
```

## 4. 특정 기준으로 정렬하기

문자열 길이를 기준으로 정렬할 수 있습니다.

```python
words = ["python", "is", "great"]

sorted_words = sorted(words, key=len)

## 10. Routing 이유까지 확인하기
  
실제 운영 환경에서는 LLM의 장황한 내부 추론을 그대로 저장하기보다는,
**Route와 짧은 판단 근거(rationale)** 정도를 구조화해서 로깅하는 편이 좋음


In [10]:
# ============================================================
# OPTIONAL. Route + 짧은 판단 근거 확인
# ============================================================

def explain_route(question: str) -> str:
    """
    수업에서 Router가 왜 특정 Route를 선택했는지
    짧은 근거를 확인하기 위한 교육용 함수 ㅈㅈ
    """

    prompt = """
당신은 AI Agent Router입니다.

사용자의 질문을 보고 다음 중 하나를 선택하세요.

CODE
DATA
WRITING
GENERAL

다음 형식만 사용하세요.

ROUTE: <하나의 Route>
REASON: <한 문장의 짧은 판단 근거>

내부 사고과정을 길게 설명하지 말고,
질문의 핵심 의도에 기반한 짧은 근거만 제시하세요.
"""

    return call_llm(
        system_prompt=prompt,
        user_prompt=question
    )


print(
    explain_route(
        "AI 프로젝트 제안서의 첫 문장을 전문적으로 작성해줘."
    )
)


ROUTE: WRITING
REASON: AI 프로젝트 제안서의 도입 문장을 전문적인 문체로 작성하는 요청입니다.


---
### Rule-based Routing
```python
if "python" in question.lower():
    route = "CODE"
```

- 개발자가 규칙을 직접 정의
- 빠르고 예측 가능
- 복잡하거나 모호한 의도 판단에는 한계

### LLM-based Routing
```python
route = call_llm(router_prompt, question)
```

- LLM이 질문의 의미와 의도를 해석
- 표현이 다양하거나 애매한 질문에도 대응 가능
- 대신 비용, latency, 오분류 가능성을 관리해야 함

### 개념적 구분

```text
Reasoning
   ↓
질문의 의미와 의도 판단

Routing
   ↓
어느 경로로 보낼지 선택

Orchestration
   ↓
선택된 경로의 컴포넌트 실행

Action
   ↓
실제 답변 생성
```
